In [20]:
import os
import wandb

import numpy as np
import random
from tqdm import *

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets
import torchvision.transforms as transforms

try:
    from torchinfo import summary
except:
    print("[INFO] Couldn't find torchinfo... installing it.")
    !pip install -q torchinfo
    from torchinfo import summary

import matplotlib.pyplot as plt
%matplotlib inline

In [21]:
def seed_everything(seed):
    random.seed(seed) # фиксируем генератор случайных чисел
    os.environ['PYTHONHASHSEED'] = str(seed) # фиксируем заполнения хешей
    np.random.seed(seed) # фиксируем генератор случайных чисел numpy
    torch.manual_seed(seed) # фиксируем генератор случайных чисел pytorch
    torch.cuda.manual_seed(seed) # фиксируем генератор случайных чисел для GPU
    torch.backends.cudnn.deterministic = True # выбираем только детерминированные алгоритмы (для сверток)
    torch.backends.cudnn.benchmark = False # фиксируем алгоритм вычисления сверток

In [22]:
class CFG:

# Задаем параметры нашего эксперимента

  api = ""
  project = "MNIST_CIFAR_convolution"
  entity = ""
  num_epochs = 20
  train_batch_size = 32
  test_batch_size = 512
  num_workers = 2
  lr = 1e-4
  seed = 42
  classes = ('airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
  wandb = False

In [23]:
# функция обучения
def train(model, device, train_loader, optimizer, criterion, epoch, WANDB):
    model.train()
    train_loss = 0
    correct = 0

    n_ex = len(train_loader)

    for batch_idx, (data, target) in tqdm(enumerate(train_loader), total=n_ex):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad() # обнуляем градиенты
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)
        correct += pred.eq(target.view_as(pred)).sum().item()
        train_loss = criterion(output, target) # считаем лосс
        train_loss.backward() # обратный проход
        optimizer.step() # делаем шаг оптимизатором

    tqdm.write('\nTrain set: Average loss: {:.4f}, Accuracy: {:.2f}%'.format(
        train_loss, 100. * correct / len(train_loader.dataset)))

    # логируем функцию потерь и точность
    if WANDB:
        wandb.log({'train_loss': train_loss,
                   'train_accuracy': correct / len(train_loader.dataset)})

In [24]:
# функция инференса
def test(model, device, test_loader, criterion, WANDB):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss = criterion(output, target)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    tqdm.write('Test set: Average loss: {:.4f}, Accuracy: {:.2f}%'.format(
        test_loss, 100. * correct / len(test_loader.dataset)))

    if WANDB:
        wandb.log({'test_loss': test_loss,
                   'test_accuracy': correct / len(test_loader.dataset)})

In [25]:
def main_CIFAR(model):

    if CFG.wandb:
        os.environ["WANDB_API_KEY"] = CFG.api
        wandb.init(project=CFG.project, entity=CFG.entity, reinit=True, config=class2dict(CFG))

    use_cuda = torch.cuda.is_available()

    seed_everything(CFG.seed)

    device = torch.device("cuda" if use_cuda else "cpu")

    kwargs = {'num_workers': CFG.num_workers, 'pin_memory': True} if use_cuda else {}

    # загружаем датасет CIFAR10
    train_loader = torch.utils.data.DataLoader(
        datasets.CIFAR10('../data', train=True, download=True,
                       transform=transforms.Compose([
                           transforms.ToTensor(),
                           transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261)) # нормализуем значения
                       ])),
        batch_size=CFG.train_batch_size, shuffle=True, **kwargs)

    test_loader = torch.utils.data.DataLoader(
        datasets.CIFAR10('../data', train=False, transform=transforms.Compose([
                           transforms.ToTensor(),
                           transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
                       ])),
        batch_size=CFG.test_batch_size, shuffle=False, **kwargs)

    model = model.to(device)


    if CFG.wandb:
        wandb.watch(model, log='all')

    optimizer = optim.Adam(model.parameters(),
                          lr=CFG.lr)

    criterion = nn.CrossEntropyLoss()

    for epoch in range(1, CFG.num_epochs + 1):
        print('\nEpoch:', epoch)
        train(model, device, train_loader, optimizer, criterion, epoch, CFG.wandb)
        test(model, device, test_loader, criterion, CFG.wandb)
    print('Training is ended!')

In [32]:
# создаем сверточную сеть для CIFAR10
class CIFAR_Net(torch.nn.Module):
    def __init__(self):
        super(CIFAR_Net, self).__init__()

        self.conv1 = torch.nn.Conv2d(3, 32, 3, padding=1)
        self.act1  = torch.nn.ReLU()
        self.pool1 = torch.nn.MaxPool2d(2, 2)

        self.conv2 = torch.nn.Conv2d(32, 64, 3, padding=1)
        self.act2  = torch.nn.ReLU()
        self.pool2 = torch.nn.MaxPool2d(2, 2)

        self.conv3 = torch.nn.Conv2d(64, 128, 3, padding=1)
        self.act3  = torch.nn.ReLU()

        self.fc1   = torch.nn.Linear(8 * 8 * 128, 256)
        self.act4  = torch.nn.ReLU()

        self.fc2   = torch.nn.Linear(256, 64)
        self.act5  = torch.nn.ReLU()

        self.fc3   = torch.nn.Linear(64, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = self.act1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.act2(x)
        x = self.pool2(x)

        x = self.conv3(x)
        x = self.act3(x)

        x = x.view(x.size(0), x.size(1) * x.size(2) * x.size(3))
        x = self.fc1(x)
        x = self.act4(x)
        x = self.fc2(x)
        x = self.act5(x)
        x = self.fc3(x)

        return x

In [33]:
seed_everything(CFG.seed)
model_CNN = CIFAR_Net()

summary(model=model_CNN,
        input_size=(32, 3, 32, 32), # входной батч
        col_names=["input_size", "output_size", "num_params", "trainable"], # что хотим посмотреть
        col_width=20
)

Layer (type:depth-idx)                   Input Shape          Output Shape         Param #              Trainable
CIFAR_Net                                [32, 3, 32, 32]      [32, 10]             --                   True
├─Conv2d: 1-1                            [32, 3, 32, 32]      [32, 32, 32, 32]     896                  True
├─ReLU: 1-2                              [32, 32, 32, 32]     [32, 32, 32, 32]     --                   --
├─MaxPool2d: 1-3                         [32, 32, 32, 32]     [32, 32, 16, 16]     --                   --
├─Conv2d: 1-4                            [32, 32, 16, 16]     [32, 64, 16, 16]     18,496               True
├─ReLU: 1-5                              [32, 64, 16, 16]     [32, 64, 16, 16]     --                   --
├─MaxPool2d: 1-6                         [32, 64, 16, 16]     [32, 64, 8, 8]       --                   --
├─Conv2d: 1-7                            [32, 64, 8, 8]       [32, 128, 8, 8]      73,856               True
├─ReLU: 1-8           

In [34]:
main_CIFAR(model_CNN)


Epoch: 1


100%|██████████| 1563/1563 [00:30<00:00, 51.11it/s]



Train set: Average loss: 1.5772, Accuracy: 42.38%
Test set: Average loss: 1.4394, Accuracy: 51.40%

Epoch: 2


100%|██████████| 1563/1563 [00:29<00:00, 53.60it/s]



Train set: Average loss: 1.1971, Accuracy: 54.13%
Test set: Average loss: 1.3222, Accuracy: 56.04%

Epoch: 3


...


Train set: Average loss: 0.1160, Accuracy: 90.82%
Test set: Average loss: 0.8479, Accuracy: 74.75%

Epoch: 19


100%|██████████| 1563/1563 [00:45<00:00, 33.99it/s]



Train set: Average loss: 0.8119, Accuracy: 92.49%
Test set: Average loss: 0.9605, Accuracy: 73.85%

Epoch: 20


100%|██████████| 1563/1563 [00:44<00:00, 35.24it/s]



Train set: Average loss: 0.2498, Accuracy: 93.40%
Test set: Average loss: 1.0335, Accuracy: 73.61%
Training is ended!
